# Pandas API Demo

This notebook exercises the pandas APIs requested in the demo list, including inspection, selection, modification, cleaning, statistics, analysis, time operations, reshaping, merge, export, and ML-style conversion.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.DataFrame({
    'date': pd.to_datetime(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05', '2024-01-06']),
    'region': ['North', 'South', 'North', 'East', 'South', 'East'],
    'channel': ['web', 'store', 'web', 'store', 'web', 'store'],
    'sales': [1200, 800, 1500, 900, 950, 1400],
    'units': [10, 8, 12, 7, 9, 11],
    'discount': [0.10, 0.05, 0.00, 0.12, 0.08, 0.15],
    'customer': ['A', 'B', 'A', 'C', 'D', 'E'],
    'status': ['ok', 'ok', 'vip', 'ok', 'vip', 'bad'],
})

print('Rows:', len(df))
print('Columns:', list(df.columns))
df.head()

Rows: 6
Columns: ['date', 'region', 'channel', 'sales', 'units', 'discount', 'customer', 'status']


,date,region,channel,sales,units,discount,customer,status
0,2024-01-01,North,web,1200,10,0.10,A,ok
1,2024-01-02,South,store,800,8,0.05,B,ok
2,2024-01-03,North,web,1500,12,0.00,A,vip
3,2024-01-04,East,store,900,7,0.12,C,ok
4,2024-01-05,South,web,950,9,0.08,D,vip


In [2]:
sales_series = df['sales']
diff_value = sales_series.diff()
pct_value = sales_series.pct_change()
shift_value = sales_series.shift(1)
rolling_value = sales_series.rolling(window=2).mean()
resample_df = df.set_index('date').resample('MS')['sales'].sum()

print('DIFF')
print(diff_value)
print('\nPCT_CHANGE')
print(pct_value)
print('\nSHIFT')
print(shift_value)
print('\nROLLING')
print(rolling_value)
print('\nRESAMPLE')
print(resample_df)

DIFF
0      NaN
1   -400.0
2    700.0
3   -600.0
4     50.0
5    450.0
Name: sales, dtype: float64

PCT_CHANGE
0         NaN
1   -0.333333
2    0.875000
3   -0.400000
4    0.055556
5    0.473684
Name: sales, dtype: float64

SHIFT
0       NaN
1    1200.0
2     800.0
3    1500.0
4     900.0
5     950.0
Name: sales, dtype: float64

ROLLING
0       NaN
1    1000.0
2    1150.0
3    1200.0
4     925.0
5    1175.0
Name: sales, dtype: float64

RESAMPLE
date
2024-01-01    6750
Freq: MS, Name: sales, dtype: int64


In [4]:
customer_info = pd.DataFrame({
    'customer': ['A', 'B', 'C', 'D', 'E'],
    'tier': ['gold', 'silver', 'gold', 'bronze', 'silver'],
})
orders = df[['customer', 'sales']].rename(columns={'sales': 'amount'})
merged = orders.merge(customer_info, on='customer', how='left')
joined = orders.set_index('customer').join(customer_info.set_index('customer'), how='left')

print('MERGE')
print(merged.head())
print('\nJOIN')
print(joined.head())

MERGE
  customer  amount    tier
0        A    1200    gold
1        B     800  silver
2        A    1500    gold
3        C     900    gold
4        D     950  bronze

JOIN
          amount    tier
customer                
A           1200    gold
B            800  silver
A           1500    gold
C            900    gold
D            950  bronze


In [5]:
output_dir = Path('data/output/pandas_api_demo')
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / 'demo.csv'
parquet_path = output_dir / 'demo.parquet'
excel_path = output_dir / 'demo.xlsx'

df.to_csv(csv_path, index=False)
print('TO_CSV')
print(csv_path.exists())

try:
    df.to_parquet(parquet_path, index=False)
    print('\nTO_PARQUET')
    print(parquet_path.exists())
except Exception as exc:
    print('\nTO_PARQUET_SKIPPED')
    print(type(exc).__name__, exc)

try:
    df.to_excel(excel_path, index=False)
    print('\nTO_EXCEL')
    print(excel_path.exists())
except Exception as exc:
    print('\nTO_EXCEL_SKIPPED')
    print(type(exc).__name__, exc)

array = df.to_numpy()
print('\nTO_NUMPY')
print(array.shape)
print(array[:2])

TO_CSV
True

TO_PARQUET_SKIPPED
ImportError Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

TO_EXCEL
True

TO_NUMPY
(6, 8)
[[Timestamp('2024-01-01 00:00:00') 'North' 'web' 1200 10 0.1 'A' 'ok']
 [Timestamp('2024-01-02 00:00:00') 'South' 'store' 800 8 0.05 'B' 'ok']]
